# Raw Data Inspection and Quality Audit

This notebook performs the first inspection of the raw Olist e-commerce dataset for the IMV customer segmentation project.

The goal of this notebook is to understand the raw tables before cleaning, feature engineering, and K-Means clustering. This stage checks table size, column structure, missing values, duplicate rows, key uniqueness, relationship consistency, and early business-relevant distributions.

This notebook does not perform final preprocessing or modeling yet.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:,.2f}".format)

In [2]:
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

raw_data_dir = project_root / "data" / "raw"

print("Project root:", project_root)
print("Raw data directory:", raw_data_dir)
print("Raw data directory exists:", raw_data_dir.exists())

Project root: d:\Workspace\Tugas_IMV_Laboratory_Recruitment\imv-customer-segmentation-kmeans
Raw data directory: d:\Workspace\Tugas_IMV_Laboratory_Recruitment\imv-customer-segmentation-kmeans\data\raw
Raw data directory exists: True


In [3]:
raw_files = {
    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
}

file_check_rows = []

for table_name, file_name in raw_files.items():
    file_path = raw_data_dir / file_name
    file_check_rows.append({
        "table_name": table_name,
        "file_name": file_name,
        "exists": file_path.exists(),
        "size_mb": file_path.stat().st_size / (1024 ** 2) if file_path.exists() else np.nan,
    })

file_check = pd.DataFrame(file_check_rows)
display(file_check)

,table_name,file_name,exists,size_mb
0,customers,olist_customers_dataset.csv,True,8.62
1,orders,olist_orders_dataset.csv,True,16.84
2,order_items,olist_order_items_dataset.csv,True,14.72
3,order_payments,olist_order_payments_dataset.csv,True,5.51
4,order_reviews,olist_order_reviews_dataset.csv,True,13.78
5,products,olist_products_dataset.csv,True,2.27
6,category_translation,product_category_name_translation.csv,True,0.00
7,geolocation,olist_geolocation_dataset.csv,True,58.44
8,sellers,olist_sellers_dataset.csv,True,0.17


In [4]:
customers = pd.read_csv(raw_data_dir / raw_files["customers"])
orders = pd.read_csv(raw_data_dir / raw_files["orders"])
order_items = pd.read_csv(raw_data_dir / raw_files["order_items"])
order_payments = pd.read_csv(raw_data_dir / raw_files["order_payments"])
order_reviews = pd.read_csv(raw_data_dir / raw_files["order_reviews"])
products = pd.read_csv(raw_data_dir / raw_files["products"])
category_translation = pd.read_csv(raw_data_dir / raw_files["category_translation"])
geolocation = pd.read_csv(raw_data_dir / raw_files["geolocation"])
sellers = pd.read_csv(raw_data_dir / raw_files["sellers"])

tables = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "products": products,
    "category_translation": category_translation,
    "geolocation": geolocation,
    "sellers": sellers,
}

print("Loaded tables:", list(tables.keys()))

Loaded tables: ['customers', 'orders', 'order_items', 'order_payments', 'order_reviews', 'products', 'category_translation', 'geolocation', 'sellers']


## Table Inventory

This section checks the number of rows, columns, and duplicate rows in each raw table.

In [5]:
inventory_rows = []

for table_name, df in tables.items():
    inventory_rows.append({
        "table_name": table_name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicate_rows": int(df.duplicated().sum()),
    })

table_inventory = pd.DataFrame(inventory_rows).sort_values("table_name").reset_index(drop=True)
display(table_inventory)

,table_name,rows,columns,duplicate_rows
0,category_translation,71,2,0
1,customers,99441,5,0
2,geolocation,1000163,5,261831
3,order_items,112650,7,0
4,order_payments,103886,5,0
5,order_reviews,99224,7,0
6,orders,99441,8,0
7,products,32951,9,0
8,sellers,3095,4,0


## Column Overview

This section lists the raw columns from each table. This helps decide which columns are identifiers, timestamps, numerical features, categorical variables, or text fields.

In [6]:
for table_name, df in tables.items():
    print(f"\n{table_name}")
    print("-" * len(table_name))
    print(list(df.columns))


customers
---------
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

orders
------
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

order_items
-----------
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

order_payments
--------------
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

order_reviews
-------------
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

products
--------
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

category_translation
--------

## Data Type Overview

This section checks the raw data types. Timestamp columns are expected to appear as object before explicit conversion.

In [7]:
dtype_rows = []

for table_name, df in tables.items():
    for column_name, dtype in df.dtypes.items():
        dtype_rows.append({
            "table_name": table_name,
            "column_name": column_name,
            "dtype": str(dtype),
        })

dtype_summary = pd.DataFrame(dtype_rows)
display(dtype_summary)

,table_name,column_name,dtype
0,customers,customer_id,str
1,customers,customer_unique_id,str
2,customers,customer_zip_code_prefix,int64
3,customers,customer_city,str
4,customers,customer_state,str
5,orders,order_id,str
6,orders,customer_id,str
7,orders,order_status,str
8,orders,order_purchase_timestamp,str
9,orders,order_approved_at,str


## Missing Value Audit

This section checks missing values in every raw table. Missing values are not automatically removed here. The goal is to understand where they occur and whether they are relevant for the segmentation workflow.

In [8]:
missing_rows = []

for table_name, df in tables.items():
    for column_name in df.columns:
        missing_count = int(df[column_name].isna().sum())
        missing_pct = missing_count / len(df) * 100
        if missing_count > 0:
            missing_rows.append({
                "table_name": table_name,
                "column_name": column_name,
                "missing_count": missing_count,
                "missing_pct": missing_pct,
            })

missing_summary = pd.DataFrame(missing_rows).sort_values(
    ["table_name", "missing_count"],
    ascending=[True, False],
).reset_index(drop=True)

display(missing_summary)

,table_name,column_name,missing_count,missing_pct
0,order_reviews,review_comment_title,87656,88.34
1,order_reviews,review_comment_message,58247,58.70
2,orders,order_delivered_customer_date,2965,2.98
3,orders,order_delivered_carrier_date,1783,1.79
4,orders,order_approved_at,160,0.16
5,products,product_category_name,610,1.85
6,products,product_name_lenght,610,1.85
7,products,product_description_lenght,610,1.85
8,products,product_photos_qty,610,1.85
9,products,product_weight_g,2,0.01


## Key Uniqueness Audit

This section checks whether important identifier columns are unique where expected.

In [9]:
key_checks = pd.DataFrame([
    {
        "table_name": "customers",
        "key": "customer_id",
        "rows": len(customers),
        "unique_values": customers["customer_id"].nunique(),
        "duplicated_keys": int(customers["customer_id"].duplicated().sum()),
    },
    {
        "table_name": "customers",
        "key": "customer_unique_id",
        "rows": len(customers),
        "unique_values": customers["customer_unique_id"].nunique(),
        "duplicated_keys": int(customers["customer_unique_id"].duplicated().sum()),
    },
    {
        "table_name": "orders",
        "key": "order_id",
        "rows": len(orders),
        "unique_values": orders["order_id"].nunique(),
        "duplicated_keys": int(orders["order_id"].duplicated().sum()),
    },
    {
        "table_name": "products",
        "key": "product_id",
        "rows": len(products),
        "unique_values": products["product_id"].nunique(),
        "duplicated_keys": int(products["product_id"].duplicated().sum()),
    },
])

display(key_checks)

,table_name,key,rows,unique_values,duplicated_keys
0,customers,customer_id,99441,99441,0
1,customers,customer_unique_id,99441,96096,3345
2,orders,order_id,99441,99441,0
3,products,product_id,32951,32951,0


## Relationship Integrity Checks

This section checks whether foreign keys in transaction tables can be linked back to their reference tables.

In [10]:
relationship_checks = pd.DataFrame([
    {
        "check": "orders.customer_id not in customers.customer_id",
        "missing_reference_count": len(set(orders["customer_id"]) - set(customers["customer_id"])),
    },
    {
        "check": "order_items.order_id not in orders.order_id",
        "missing_reference_count": len(set(order_items["order_id"]) - set(orders["order_id"])),
    },
    {
        "check": "order_payments.order_id not in orders.order_id",
        "missing_reference_count": len(set(order_payments["order_id"]) - set(orders["order_id"])),
    },
    {
        "check": "order_reviews.order_id not in orders.order_id",
        "missing_reference_count": len(set(order_reviews["order_id"]) - set(orders["order_id"])),
    },
    {
        "check": "order_items.product_id not in products.product_id",
        "missing_reference_count": len(set(order_items["product_id"]) - set(products["product_id"])),
    },
    {
        "check": "order_items.seller_id not in sellers.seller_id",
        "missing_reference_count": len(set(order_items["seller_id"]) - set(sellers["seller_id"])),
    },
    {
        "check": "products.product_category_name not in category_translation.product_category_name",
        "missing_reference_count": len(
            set(products["product_category_name"].dropna()) - set(category_translation["product_category_name"])
        ),
    },
])

display(relationship_checks)

,check,missing_reference_count
0,orders.customer_id not in customers.customer_id,0
1,order_items.order_id not in orders.order_id,0
2,order_payments.order_id not in orders.order_id,0
3,order_reviews.order_id not in orders.order_id,0
4,order_items.product_id not in products.product_id,0
5,order_items.seller_id not in sellers.seller_id,0
6,products.product_category_name not in category...,2


In [11]:
missing_translations = sorted(
    set(products["product_category_name"].dropna()) - set(category_translation["product_category_name"])
)

missing_translations

['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']

## Order Status and Date Range

This section checks order status distribution and purchase timestamp range. This helps decide which orders should be included in customer segmentation.

In [12]:
order_status_counts = orders["order_status"].value_counts().rename_axis("order_status").reset_index(name="order_count")
display(order_status_counts)

,order_status,order_count
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [13]:
orders_dates = orders.copy()

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for column in date_columns:
    orders_dates[column] = pd.to_datetime(orders_dates[column], errors="coerce")

print("Purchase date min:", orders_dates["order_purchase_timestamp"].min())
print("Purchase date max:", orders_dates["order_purchase_timestamp"].max())

Purchase date min: 2016-09-04 21:15:19
Purchase date max: 2018-10-17 17:30:18


In [14]:
missing_delivery_by_status = (
    orders_dates
    .groupby("order_status")["order_delivered_customer_date"]
    .apply(lambda series: int(series.isna().sum()))
    .reset_index(name="missing_delivered_customer_date")
)

display(missing_delivery_by_status)

,order_status,missing_delivered_customer_date
0,approved,2
1,canceled,619
2,created,5
3,delivered,8
4,invoiced,314
5,processing,301
6,shipped,1107
7,unavailable,609


## Payment Audit

This section checks payment method distribution and numerical ranges for payment features.

In [15]:
payment_type_counts = (
    order_payments["payment_type"]
    .value_counts()
    .rename_axis("payment_type")
    .reset_index(name="payment_count")
)

display(payment_type_counts)

,payment_type,payment_count
0,credit_card,76795
1,boleto,19784
2,voucher,5775
3,debit_card,1529
4,not_defined,3


In [16]:
display(order_payments[["payment_installments", "payment_value"]].describe())

,payment_installments,payment_value
count,"103,886.00","103,886.00"
mean,2.85,154.10
std,2.69,217.49
min,0.00,0.00
25%,1.00,56.79
50%,1.00,100.00
75%,4.00,171.84
max,24.00,"13,664.08"


## Review Score Audit

This section checks the review score distribution. Review comments may be missing, but review score is the main customer satisfaction signal planned for this project.

In [17]:
review_score_counts = (
    order_reviews["review_score"]
    .value_counts()
    .sort_index()
    .rename_axis("review_score")
    .reset_index(name="review_count")
)

display(review_score_counts)

,review_score,review_count
0,1,11424
1,2,3151
2,3,8179
3,4,19142
4,5,57328


## Item Price and Freight Audit

This section checks numerical ranges for item price and freight value.

In [18]:
display(order_items[["price", "freight_value"]].describe())

,price,freight_value
count,"112,650.00","112,650.00"
mean,120.65,19.99
std,183.63,15.81
min,0.85,0.00
25%,39.90,13.08
50%,74.99,16.26
75%,134.90,21.15
max,"6,735.00",409.68


## Orders Without Related Records

This section checks how many orders do not have linked item, payment, or review records. This matters because feature engineering will join multiple tables.

In [19]:
orders_without_related_records = pd.DataFrame([
    {
        "condition": "orders without order_items",
        "order_count": len(set(orders["order_id"]) - set(order_items["order_id"])),
    },
    {
        "condition": "orders without order_payments",
        "order_count": len(set(orders["order_id"]) - set(order_payments["order_id"])),
    },
    {
        "condition": "orders without order_reviews",
        "order_count": len(set(orders["order_id"]) - set(order_reviews["order_id"])),
    },
])

display(orders_without_related_records)

,condition,order_count
0,orders without order_items,775
1,orders without order_payments,1
2,orders without order_reviews,768


## Preliminary Raw Data Findings

The raw Olist dataset is suitable for the planned customer segmentation project. The main customer, order, item, payment, review, and product tables are available and can be linked through their identifier columns.

Important findings from this raw inspection:

- The dataset contains 99,441 order records and 96,096 unique customer identities.
- The delivered order status is dominant, but non-delivered statuses also exist.
- Delivery timestamp columns contain missing values, especially for non-delivered orders.
- Review text fields contain many missing values, but review score is available and useful for customer satisfaction analysis.
- Product category fields have some missing values.
- The geolocation table contains many duplicate rows and should be treated carefully if used.
- Payment and item value columns are likely skewed and should be inspected before K-Means modeling.
- Raw IDs should be used for joins, not as clustering features.

The next progress should focus on cleaning decisions and building a reliable customer-level analytical dataset.